This notebook shows a model predicting medicare payment amount bease on volume and risk.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

file_path = "../data/raw/Modified_Prov_Type_and_Round_Int_All_10_years.csv"

df = pd.read_csv(file_path, nrows=100000)
df.columns.tolist()
df["log_Tot_Benes"] = np.log1p(df["Tot_Benes"])
df["log_Tot_Srvcs"] = np.log1p(df["Tot_Srvcs"])
df["Services_per_Bene"] = df["Tot_Srvcs"] / df["Tot_Benes"].replace(0, np.nan)
df["Drug_Service_Share"] = df["Drug_Tot_Srvcs"] / df["Tot_Srvcs"].replace(0, np.nan)
df["log_Tot_Mdcr_Pymt_Amt"] = np.log1p(df["Tot_Mdcr_Pymt_Amt"])

C:\Users\alybo\AppData\Local\Temp\ipykernel_37212\3146712888.py:9: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, nrows=100000)


['Unnamed: 0',
 'Rndrng_NPI',
 'Rndrng_Prvdr_Last_Org_Name',
 'Rndrng_Prvdr_First_Name',
 'Rndrng_Prvdr_MI',
 'Rndrng_Prvdr_Crdntls',
 'Rndrng_Prvdr_Ent_Cd',
 'Rndrng_Prvdr_St1',
 'Rndrng_Prvdr_St2',
 'Rndrng_Prvdr_City',
 'Rndrng_Prvdr_State_Abrvtn',
 'Rndrng_Prvdr_State_FIPS',
 'Rndrng_Prvdr_Zip5',
 'Rndrng_Prvdr_RUCA',
 'Rndrng_Prvdr_RUCA_Desc',
 'Rndrng_Prvdr_Cntry',
 'Rndrng_Prvdr_Type',
 'Rndrng_Prvdr_Mdcr_Prtcptg_Ind',
 'Tot_HCPCS_Cds',
 'Tot_Benes',
 'Tot_Srvcs',
 'Tot_Sbmtd_Chrg',
 'Tot_Mdcr_Alowd_Amt',
 'Tot_Mdcr_Pymt_Amt',
 'Tot_Mdcr_Stdzd_Amt',
 'Drug_Sprsn_Ind',
 'Drug_Tot_HCPCS_Cds',
 'Drug_Tot_Benes',
 'Drug_Tot_Srvcs',
 'Drug_Sbmtd_Chrg',
 'Drug_Mdcr_Alowd_Amt',
 'Drug_Mdcr_Pymt_Amt',
 'Drug_Mdcr_Stdzd_Amt',
 'Med_Sprsn_Ind',
 'Med_Tot_HCPCS_Cds',
 'Med_Tot_Benes',
 'Med_Tot_Srvcs',
 'Med_Sbmtd_Chrg',
 'Med_Mdcr_Alowd_Amt',
 'Med_Mdcr_Pymt_Amt',
 'Med_Mdcr_Stdzd_Amt',
 'Bene_Avg_Age',
 'Bene_Age_LT_65_Cnt',
 'Bene_Age_65_74_Cnt',
 'Bene_Age_75_84_Cnt',
 'Bene_Age_GT_84_

In [ ]:
min_count = 500

keep_types = type_counts[type_counts >= min_count].index

df["ProvType_Model"] = df["Rndrng_Prvdr_Type"].where(
    df["Rndrng_Prvdr_Type"].isin(keep_types),
    "Other"
)

These cells are just looking at the dataset as a whole and checking that provider types are organized into sub catagories

## Now a baseline linear regression model

In [ ]:
train = df[df["Year"] <= 2020].copy()
test  = df[df["Year"] >= 2021].copy()

features = [
    "log_Tot_Benes",
    "log_Tot_Srvcs",
    "Services_per_Bene",
    "Drug_Service_Share",
    "Bene_Avg_Risk_Scre",
    "ProvType_Model"
]

X_train = train[features]
y_train = train["log_Tot_Mdcr_Pymt_Amt"]

X_test = test[features]
y_test = test["log_Tot_Mdcr_Pymt_Amt"]

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error